<a href="https://colab.research.google.com/github/yasminandrade0322-hue/ATIVIDADES_DEV_PYTHON/blob/main/atv_streamlit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
# Instalar o Streamlit
!pip install streamlit pandas numpy

# Criar um arquivo e rodar
!echo "import streamlit as st; st.write('Olá, mundo!')" > app.py
!streamlit run app.py




  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://104.197.100.18:8501

  Stopping...
  Stopping...


In [5]:
import streamlit as st
import pandas as pd
import numpy as np
import plotly.express as px

# Configuração
st.set_page_config(
    page_title="Dashboard de Funcionários",
    layout="wide"
)

st.title("📊 Dashboard de Análise de Funcionários")

# Gerando dados (ou carregar CSV)
@st.cache_data
def carregar_dados():
    dados = {
        "nome": ["Ana", "Bruno", "Carlos", "Daniela", "Eduardo"],
        "idade": [23, 35, 29, np.nan, 40],
        "cidade": ["SP", "RJ", "SP", "MG", "RJ"],
        "salario": [3000, 5000, 4000, 3500, np.nan],
        "data_contratacao": pd.to_datetime([
            "2020-01-10", "2019-03-15", "2021-07-22", "2018-11-30", "2022-05-10"
        ])
    }

    df = pd.DataFrame(dados)

    # Limpeza
    df["idade"] = df["idade"].fillna(df["idade"].mean())
    df["salario"] = df["salario"].fillna(df["salario"].median())

    # Feature engineering
    df["salario_anual"] = df["salario"] * 12
    df["ano_contratacao"] = df["data_contratacao"].dt.year

    df["categoria_salario"] = df["salario"].apply(
        lambda x: "Alto" if x > 4500 else "Médio" if x > 3000 else "Baixo"
    )

    return df

df = carregar_dados()

# SIDEBAR (FILTROS)
st.sidebar.header("🔎 Filtros")

cidades = st.sidebar.multiselect(
    "Selecione a cidade",
    options=df["cidade"].unique(),
    default=df["cidade"].unique()
)

faixa_salario = st.sidebar.slider(
    "Faixa salarial",
    float(df["salario"].min()),
    float(df["salario"].max()),
    (float(df["salario"].min()), float(df["salario"].max()))
)

df_filtrado = df[
    (df["cidade"].isin(cidades)) &
    (df["salario"] >= faixa_salario[0]) &
    (df["salario"] <= faixa_salario[1])
]

# Desafio 1 - fácil
categoria_salario = st.sidebar.selectbox(
    "Categoria salário",
    options=["Todas"] + list(df["categoria_salario"].unique())
)
if categoria_salario !="Todas":
  df_filtrado = df_filtrado[
      df_filtrado["categoria_salario"] == categoria_salario
  ]

# KPIs
col1, col2, col3 = st.columns(3)

col1.metric("💰 Salário Médio", f"R$ {df_filtrado['salario'].mean():.2f}")
col2.metric("👥 Total Funcionários", df_filtrado.shape[0])
col3.metric("📈 Salário Máximo", f"R$ {df_filtrado['salario'].max():.2f}")

# TABELA
st.subheader("📋 Dados")
st.dataframe(df_filtrado, use_container_width=True)

# GRÁFICOS
st.subheader("📊 Análises")

col1, col2 = st.columns(2)

# Média salarial por cidade
media_cidade = df_filtrado.groupby("cidade")["salario"].mean()

# Desafio 2 - médio
media_cidade_df = df_filtrado.groupby(["cidade", "categoria_salario"])["salario"].mean().reset_index()

fig1 = px.bar(
    media_cidade_df,
    x="cidade",
    y="salario",
    color="categoria_salario",
    barmode="group",
    title="Média Salarial por Cidade",
    hover_data={
        "cidade": True,
        "salario": ":.2f",
        "categoria_salario": True
    },
    color_discrete_map={
        "Alto": "green",
        "Médio": "orange",
        "Baixo": "red"
    }
)

col1.plotly_chart(fig1, use_container_width=True)

# Distribuição por categoria - desafio 2
categoria_df = df_filtrado["categoria_salario"].value_counts().reset_index()
categoria_df.columns = ["categoria_salario", "quantidade"]

fig2 = px.bar(
    categoria_df,
    x="categoria_salario",
    y="quantidade",
    color="categoria_salario",
    title="Distribuição por Categoria Salarial",
    hover_data={
        "categoria_salario": True,
        "quantidade": True
    },
    color_discrete_map={
        "Alto": "green",
        "Médio": "orange",
        "Baixo": "red"
    }
)

col2.plotly_chart(fig2, use_container_width=True)

# PIVOT TABLE
st.subheader("📌 Tabela Dinâmica")

pivot = pd.pivot_table(
    df_filtrado,
    values="salario",
    index="cidade",
    columns="categoria_salario",
    aggfunc="mean"
)

st.dataframe(pivot)

# DOWNLOAD
st.subheader("⬇️ Download dos dados")

csv = df_filtrado.to_csv(index=False).encode("utf-8")
st.download_button(
    label="Baixar CSV",
    data=csv,
    file_name="dados_filtrados.csv",
    mime="text/csv"
)

# UPLOAD DE ARQUIVO
st.sidebar.subheader("📂 Upload de CSV")

uploaded_file = st.sidebar.file_uploader("Envie um CSV", type=["csv"])

if uploaded_file:
    df_upload = pd.read_csv(uploaded_file)
    st.write("📄 Dados carregados:")
    st.dataframe(df_upload)

2026-04-07 23:05:47.784 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-07 23:05:47.786 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-07 23:05:47.787 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-07 23:05:47.787 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-07 23:05:47.789 No runtime found, using MemoryCacheStorageManager
2026-04-07 23:05:47.793 No runtime found, using MemoryCacheStorageManager
2026-04-07 23:05:47.794 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-07 23:05:47.822 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-07 23:05:47.823 Thread 'MainThread': missing ScriptRunContext! This warning can be ignor